# Map/Shuffle/Reduce, Combiners, Immutable Transformations, Join Strategies, Salting

## What you'll practice

Before touching Spark, *be* Spark: split work into partitions, transform without mutating, aggregate locally before shuffling, and choose a join strategy. Each exercise is a tiny simulation
of something Spark does for you — so that `explain()` output, shuffle sizes and skew make sense when you see them for real.

> Function and class names in **Implement** are what the tests look for. Run all tests for this notebook with `pytest tests/pyspark_prep/test_12_01_functional_partition_thinking.py -v` from the lab root. Untouched exercises show as *skipped*.
>
> Solutions: `solutions/12_pyspark_preparation/01_functional_partition_thinking.ipynb`. Don't open it until you've tried.

## Exercise 1 — 🟢 Easy · Word count as explicit map → shuffle → reduce

**Implement:** `map_phase(lines) -> Iterator[tuple[str, int]]`, `shuffle_phase(pairs) -> dict[str, list[int]]`, `reduce_phase(groups) -> dict[str, int]`

`map_phase` emits `(word, 1)` for each lowercase word (split on whitespace, strip `.,!?`); `shuffle_phase` groups values by key; `reduce_phase` sums each group. Chaining the three gives word counts.

**Input:** `lines`

**Expected output:** `Iterator[tuple[str, int]]`

**Constraints:** Standard library only (the last notebook uses the PySpark DataFrame API).

**Example:** the tests are executable examples — read `test_word_count_phases` in `tests/pyspark_prep/test_12_01_functional_partition_thinking.py`.

**Check:** `pytest tests/pyspark_prep/test_12_01_functional_partition_thinking.py -k test_word_count_phases`

**YOUR CODE ↓**

## Exercise 2 — 🟡 Medium · Combiners shrink the shuffle

**Implement:** `partitioned_sum(records: list[dict], key: str, value: str, num_partitions: int, use_combiner: bool = True) -> tuple[dict, int]`

Simulate a cluster: split `records` round-robin into `num_partitions` input partitions. With `use_combiner`, each partition first sums values per key locally (map-side combine) and emits one
`(key, partial)` per key; without it, every record is emitted as `(key, value)`. Count emitted pairs = **shuffled records**. Then route pairs to reducers by `zlib.crc32(key) % num_partitions` and sum.
Return `(totals_dict, shuffled_records)`. The totals must be identical in both modes.

**Input:** `records: list[dict], key: str, value: str, num_partitions: int, use_combiner: bool = True`

**Expected output:** `tuple[dict, int]`

**Constraints:** Standard library only (the last notebook uses the PySpark DataFrame API).

**Example:** the tests are executable examples — read `test_partitioned_sum` in `tests/pyspark_prep/test_12_01_functional_partition_thinking.py`.

**Check:** `pytest tests/pyspark_prep/test_12_01_functional_partition_thinking.py -k test_partitioned_sum`

**YOUR CODE ↓**

## Exercise 3 — 🟡 Medium · Immutable DataFrame-style transformations

**Implement:** `transform(rows: list[dict], steps: list[tuple]) -> list[dict]`

Apply steps like Spark DataFrame methods, never mutating the input rows: `("with_column", name, fn)` (fn receives the row), `("filter", predicate)`, `("select", [cols])`,
`("drop", [cols])`, `("rename", {old: new})`. Unknown step → `ValueError`. Selecting a missing column → `KeyError` (Spark raises `AnalysisException` at plan time).

**Input:** `rows: list[dict], steps: list[tuple]`

**Expected output:** `list[dict]`

**Constraints:** Standard library only (the last notebook uses the PySpark DataFrame API).

**Example:** the tests are executable examples — read `test_transform` in `tests/pyspark_prep/test_12_01_functional_partition_thinking.py`.

**Check:** `pytest tests/pyspark_prep/test_12_01_functional_partition_thinking.py -k test_transform`

**YOUR CODE ↓**

## Exercise 4 — 🟠 Hard · Broadcast join vs shuffle join

**Implement:** `broadcast_join(large: list[dict], small: list[dict], key: str) -> list[dict]`, `shuffle_join(left: list[dict], right: list[dict], key: str, n: int) -> tuple[list[dict], list[int]]`

Inner joins two ways. **Broadcast**: build a dict from the small side (key → list of rows), stream the large side — no shuffle of the large table. **Shuffle**: hash-partition *both*
sides by key into `n` partitions (crc32), then join each partition pair independently with a local hash join; also return the per-partition output sizes. Joined rows are `{**left, **right}`
(right wins on conflicts). Both functions must produce the same multiset of rows.

**Input:** `large: list[dict], small: list[dict], key: str`

**Expected output:** `list[dict]`

**Constraints:** Standard library only (the last notebook uses the PySpark DataFrame API).

**Example:** the tests are executable examples — read `test_joins_agree` in `tests/pyspark_prep/test_12_01_functional_partition_thinking.py`.

**Check:** `pytest tests/pyspark_prep/test_12_01_functional_partition_thinking.py -k test_joins_agree`

**YOUR CODE ↓**

## Challenge — 🔴 Very Hard · Fix a skewed join with salting · 🧩 Mini-project

**Implement:** `salted_join(left: list[dict], right: list[dict], key: str, n: int, hot_keys: set, salt_buckets: int = 8) -> tuple[list[dict], list[int]]`

One customer has 90% of the orders: in a shuffle join, one partition gets almost everything (a straggler task). **Salting**: for rows whose key is in `hot_keys`, give each **left** row a salt
`zlib.crc32(repr(sorted(row.items())).encode()) % salt_buckets` and **replicate** each matching right row once per salt value; partition by `(key, salt)`. Non-hot keys use salt 0 and aren't
replicated. Return `(joined_rows, per_partition_output_sizes)`. Results must equal a normal join; the max partition size should drop substantially. Compare with `shuffle_join` on the skewed data.

**Input:** `left: list[dict], right: list[dict], key: str, n: int, hot_keys: set, salt_buckets: int = 8`

**Expected output:** `tuple[list[dict], list[int]]`

**Constraints:** Standard library only (the last notebook uses the PySpark DataFrame API).

**Example:** the tests are executable examples — read `test_salted_join` in `tests/pyspark_prep/test_12_01_functional_partition_thinking.py`.

**Check:** `pytest tests/pyspark_prep/test_12_01_functional_partition_thinking.py -k test_salted_join`

**YOUR CODE ↓**